In [ ]:
!pip install openai-whisper transformers soundfile accelerate ffmpeg-python

In [ ]:
!pip install faiss-cpu sentence-transformers

In [ ]:
import whisper
import torch
import soundfile as sf
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display, Audio

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading English Models...")

stt_model = whisper.load_model("medium.en", device=device)

llm_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_id)
llm_model = AutoModelForCausalLM.from_pretrained(llm_id, torch_dtype=torch.bfloat16).to(device)

tts_model, _ = torch.hub.load(
    repo_or_dir='snakers4/silero-models',
    model='silero_tts',
    language='en',
    speaker='v3_en',
    trust_repo=True
)
tts_model.to(device)
print("Models loaded successfully!")

In [ ]:
from IPython.display import HTML, display
from google.colab.output import eval_js
from base64 import b64decode
import subprocess

AUDIO_HTML = """
<script>
var my_div = document.createElement("DIV");
var my_btn = document.createElement("BUTTON");
var t = document.createTextNode("Click to Start Recording");

my_btn.appendChild(t);
my_btn.style.padding = "10px";
my_btn.style.fontSize = "16px";
my_btn.style.color = "white";
my_btn.style.backgroundColor = "#4CAF50";
my_btn.style.border = "none";
my_btn.style.borderRadius = "5px";
my_btn.style.cursor = "pointer";
my_div.appendChild(my_btn);
document.body.appendChild(my_div);

var resolve_py;
window.audioPromise = new Promise(resolve => {
    resolve_py = resolve;
});

var recorder, gumStream;

my_btn.onclick = function() {
    if (my_btn.innerText === "Click to Start Recording") {
        navigator.mediaDevices.getUserMedia({audio: true}).then(stream => {
            gumStream = stream;
            recorder = new MediaRecorder(stream);
            var chunks = [];

            recorder.ondataavailable = e => chunks.push(e.data);

            recorder.onstop = () => {
                var blob = new Blob(chunks, { 'type' : 'audio/webm' });
                var reader = new FileReader();
                reader.readAsDataURL(blob);
                reader.onloadend = () => {
                    resolve_py(reader.result);
                }
            };
            recorder.start();
            my_btn.innerText = "Recording... Click to Stop";
            my_btn.style.backgroundColor = "#f44336";
        }).catch(err => {
            alert("Microphone access denied: " + err.message);
        });
    } else if (my_btn.innerText === "Recording... Click to Stop") {
        recorder.stop();
        gumStream.getAudioTracks()[0].stop();
        my_div.remove();
    }
};
</script>
"""

def record_audio_colab(filename='input.wav'):
    display(HTML(AUDIO_HTML))
    data = eval_js("window.audioPromise")

    if not isinstance(data, str) or ',' not in data:
        print("[System Error] No valid audio data received.")
        return False

    binary = b64decode(data.split(',')[1])

    with open("input.webm", "wb") as f:
        f.write(binary)

    subprocess.run(["ffmpeg", "-y", "-i", "input.webm", filename], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("Audio saved successfully!")
    return True

In [ ]:
import sqlite3
import faiss
import numpy as np
import logging
import torch

class DatabaseManager:
    def __init__(self, embedder):
        self.embedder = embedder
        self.conn = sqlite3.connect(':memory:', check_same_thread=False)
        self.cursor = self.conn.cursor()
        self._setup_sql()
        self._setup_faiss()

    def _setup_sql(self):
        self.cursor.execute('''CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, name TEXT, password TEXT, balance REAL, type TEXT, status TEXT)''')
        users = [
            (101, 'Alice', '1234', 5430.00, 'Standard', 'Active'),
            (102, 'Bob', '1234', 25000.50, 'Premium', 'Active'),
            (103, 'Charlie', '1234', 0.00, 'Standard', 'Blocked')
        ]
        self.cursor.executemany("INSERT INTO accounts VALUES (?,?,?,?,?,?)", users)

    def _setup_faiss(self):
        self.rules = [
            "Lost cards must be reported to 1-800-SAFE-BANK immediately.",
            "Housing loan interest rate is 18%.",
            "To block a lost card, use the mobile app or call 1-800-BANK immediately.",
            "Housing loans have an 18% interest rate and require a property deed as collateral.",
            "To open a new account, you need a National ID, proof of address, and a minimum $50 deposit.",
            "International transfers take 3-5 business days and incur a flat $15 fee."
        ]
        embeddings = self.embedder.encode(self.rules)
        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(np.array(embeddings))

    def authenticate(self, username, password):
        self.cursor.execute("SELECT * FROM accounts WHERE name=? AND password=?", (username, password))
        return self.cursor.fetchone()

    def search_faiss(self, query, threshold=0.8):
        q_emb = self.embedder.encode([query])
        distances, indices = self.index.search(np.array(q_emb), k=1)

        if distances[0][0] > threshold:
            return None
        return self.rules[indices[0][0]]

class BankingAgent:
    def __init__(self, db_manager, models, device):
        self.db = db_manager
        self.stt, self.tts, self.llm, self.tokenizer = models
        self.device = device
        self.logger = logging.getLogger(__name__)
        self.memory = []
        self.current_user = None

    def login(self, u, p):
        user = self.db.authenticate(u, p)
        if user:
            self.current_user = {"name": user[1], "balance": user[3], "type": user[4], "status": user[5]}
            return True
        return False

    def route_query(self, query):
        q_low = query.lower()
        if "balance" in q_low:
            return f"Your current balance is ${self.current_user['balance']}."

        rule = self.db.search_faiss(query)
        if rule is None:
            return "NO_RELIABLE_INFO"
        return rule

    def generate_response(self, user_input):
        context = self.route_query(user_input)

        if context == "NO_RELIABLE_INFO":
            reply = "I don't have reliable information on that."
            self.memory.append(f"U: {user_input} | A: {reply}")
            return reply

        messages = [
            {"role": "system", "content": "You are a banking assistant. You must answer ONLY using the provided Fact. If the Fact contains a phone number or specific instruction, include it."},
            {"role": "user", "content": f"Fact: {context}\n\nUser Question: {user_input}"}
        ]

        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.llm.generate(
                **inputs,
                max_new_tokens=50,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )

        reply = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

        if "Assistant:" in reply:
            reply = reply.split("Assistant:")[-1].strip()
        if "User:" in reply:
            reply = reply.split("User:")[0].strip()

        self.memory.append(f"U: {user_input} | A: {reply}")
        return reply

In [ ]:
import getpass
import soundfile as sf
import torch
from IPython.display import display, Audio
from sentence_transformers import SentenceTransformer

print("Loading Embedding Model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

models_tuple = (stt_model, tts_model, llm_model, llm_tokenizer)

db = DatabaseManager(embedder)
device = "cuda" if torch.cuda.is_available() else "cpu"
bank_agent = BankingAgent(db, models_tuple, device)

print("\n" + "="*40)
print("   ENTERPRISE MULTIMODAL BANKING AI   ")
print("="*40)

while True:
    u = input("\nUsername: ").strip().capitalize()
    p = getpass.getpass("Password: ").strip()

    if bank_agent.login(u, p):
        print(f"\n[System] Welcome {u}! Logging events to bank_agent.log")
        break
    else:
        print("\n[Error] Invalid Credentials. Please try again.")

while True:
    print("\nHow would you like to communicate?")
    mode = input("Type 't' for Text, 'v' for Voice, or 'exit' to quit: ").strip().lower()

    if mode == 'exit':
        print("Terminating session. Goodbye!")
        break

    query = ""

    if mode == 'v':
        print("\n[Voice Mode] Ready. Please speak and then click the stop button.")
        success = record_audio_colab()

        if not success:
            continue

        print("Transcribing...")
        transcription = stt_model.transcribe(
            "input.wav",
            fp16=torch.cuda.is_available(),
            initial_prompt="bank, account, current balance, money, loan, finance, lost card."
        )
        query = transcription["text"].strip()
        print(f"User (Voice): {query}")

    elif mode == 't':
        query = input("\nUser (Text): ").strip()

    else:
        print("[Error] Invalid selection.")
        continue

    if query:
        response = bank_agent.generate_response(query)
        print(f"Agent: {response}")

        audio = tts_model.apply_tts(text=response, speaker='en_0', sample_rate=24000)
        sf.write("output.wav", audio.cpu().numpy(), 24000)
        display(Audio("output.wav", autoplay=True))